# 📊 Biais et approfondissement

> ✍️ **Notebook dirigé** · complétez les cellules marquées ✍️ À VOUS.

**Jour 1 · 14h15** · Exercice

*Analyse exploratoire et interprétation métier*

**Objectifs**
- Détecter les sur- et sous-représentations par rapport à une référence
- Quantifier les disparités d'une décision algorithmique avec Fairlearn
- Identifier les corrélations induites et les limites de l'analyse

In [2]:
%pip install -q fairlearn
# ⚙️ À exécuter en premier : récupère la configuration du cours (quelques secondes)
import os, sys
CONFIG_ID = "1bW5RsToUHc-7pGiyHS_1B2OYkAFsrv3r"

# Suppression du fichier corrompu s'il existe
if os.path.exists("config_projet.py"):
    os.remove("config_projet.py")

# Téléchargement propre avec gdown pour contourner l'avertissement de virus Google Drive
!gdown --id {CONFIG_ID} -O config_projet.py

print("✅ Configuration prête : chaque fichier de données se télécharge au premier usage")


Note: you may need to restart the kernel to use updated packages.
✅ Configuration prête : chaque fichier de données se télécharge au premier usage


usage: gdown [-h] [-V] [-O OUTPUT] [-q] [--proxy PROXY] [--speed SPEED]
             [--no-cookies] [--no-check-certificate] [--continue] [--folder]
             [--format FORMAT] [--user-agent USER_AGENT]
             url_or_id
gdown: error: unrecognized arguments: --id


In [4]:
# Choisissez votre secteur : "commerce", "banque" ou "hotellerie"
SECTEUR = "banque"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from config_projet import charger, reference, nettoyage_minimal

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)
brut, cfg = charger(SECTEUR)
print(cfg["nom"], "·", brut.shape[0], "lignes,", brut.shape[1], "colonnes")

Crédit Azur · 20477 lignes, 20 colonnes


In [5]:
from fairlearn.metrics import (MetricFrame, selection_rate, count, true_positive_rate,
                               demographic_parity_difference, demographic_parity_ratio)
df = nettoyage_minimal(brut, cfg)
for col in cfg["sensibles"]:
    df[col] = df[col].fillna("inconnu")
y = df[cfg["decision"]]
print("Décision étudiée :", cfg["decision_label"], f"· taux global {y.mean():.1%}")

Décision étudiée : crédit accordé · taux global 64.0%


**Commentaire :**

Le jeu de données Crédit Azur contient 20 477 demandes de crédit réparties sur 20 variables. L'analyse porte sur la décision d'octroi du crédit (`crédit accordé`). Le taux global d'accord est de 64 %, ce qui signifie qu'environ 64 % des demandes de crédit ont reçu une réponse favorable.

Ce taux global servira de référence pour comparer les décisions entre les différents groupes de clients et rechercher d'éventuelles disparités.

## 1. La représentativité
Sur les **clients uniques**, comparez la part de chaque modalité à la population de référence (`reference()`). Signalez les ratios inférieurs à 0,8 ou supérieurs à 1,25.

In [6]:

# 1. Comparaison des clients à la population de référence

# On conserve une seule ligne par client
df_clients = df.dropna(subset=["client_id"]).drop_duplicates(
    subset=["client_id"]
).copy()

print("Nombre de clients uniques :", len(df_clients))

# Analyse de chaque variable sensible
for col in cfg["sensibles"]:

    print("\n" + "=" * 60)
    print("VARIABLE ÉTUDIÉE :", col)

    # Répartition des clients par modalité
    effectifs = df_clients[col].fillna("inconnu").value_counts()

    taux_clients = effectifs / len(df_clients) * 100

    # Récupération de la référence correspondant à la variable
    ref_variable = ref[ref["variable"] == col].copy()

    reference_taux = (
        ref_variable.set_index("modalite")["part_reference"] * 100
    )

    # Regroupement des résultats dans un tableau
    comparaison = pd.DataFrame({
        "Effectif clients": effectifs,
        "Clients (%)": taux_clients,
        "Référence (%)": reference_taux
    }).fillna(0)

    # Calcul de l'écart en points de pourcentage
    comparaison["Écart (points)"] = (
        comparaison["Clients (%)"] - comparaison["Référence (%)"]
    )

    # Calcul du ratio de représentativité
    comparaison["Ratio"] = (
        comparaison["Clients (%)"] / comparaison["Référence (%)"]
    ).where(comparaison["Référence (%)"] != 0)

    display(comparaison.round(2))


Nombre de clients uniques : 11170

VARIABLE ÉTUDIÉE : tranche_age


,Effectif clients,Clients (%),Référence (%),Écart (points),Ratio
18-24,1017,9.10,10.0,-0.90,0.91
25-34,2905,26.01,15.0,11.01,1.73
35-44,2847,25.49,16.0,9.49,1.59
45-54,2185,19.56,17.0,2.56,1.15
55-64,1450,12.98,16.0,-3.02,0.81
65+,766,6.86,26.0,-19.14,0.26



VARIABLE ÉTUDIÉE : genre


,Effectif clients,Clients (%),Référence (%),Écart (points),Ratio
F,4784,42.83,52.0,-9.17,0.82
H,6386,57.17,48.0,9.17,1.19



VARIABLE ÉTUDIÉE : region


,Effectif clients,Clients (%),Référence (%),Écart (points),Ratio
Nord-Est,2611,23.38,24.0,-0.62,0.97
Ouest,2341,20.96,21.0,-0.04,1.00
Outre-mer,118,1.06,3.0,-1.94,0.35
Sud-Est,2468,22.09,19.0,3.09,1.16
Sud-Ouest,1551,13.89,14.0,-0.11,0.99
Île-de-France,2081,18.63,19.0,-0.37,0.98


### Interprétation – Représentativité

**Tranche d'âge :** Les 25-34 ans et les 35-44 ans sont surreprésentés, avec des ratios de 1,73 et 1,59. À l'inverse, les 65 ans et plus sont fortement sous-représentés (0,26).

**Genre :** Les hommes sont plus représentés (57,17 %) que les femmes (42,83 %). Cependant, les ratios restent entre 0,8 et 1,25, donc aucun groupe ne présente de déséquilibre majeur selon les seuils retenus.

**Région :** La répartition est globalement équilibrée, sauf pour l'Outre-mer, qui est fortement sous-représenté avec un ratio de 0,35.

**Conclusion :** On observe des différences de représentativité selon l'âge, le genre et la région. Cependant, ces écarts ne prouvent pas une discrimination dans l'octroi des crédits. Il faut maintenant comparer les taux d'accord entre les groupes.

## 2. La parité de la décision
Pour chaque variable sensible, calculez avec `MetricFrame` le taux de décision favorable et l'effectif par groupe, puis l'écart et le ratio de parité démographique.

In [7]:

# 2. La parité de la décision

synthese = []

# Variable représentant la décision de crédit
y = df[cfg["decision"]]

for col in cfg["sensibles"]:

    # Identification des groupes de clients
    groupes = df[col].fillna("inconnu")

    # Calcul des indicateurs par groupe avec MetricFrame
    mf = MetricFrame(
        metrics={
            "Effectif": count,
            "Taux d'accord": selection_rate
        },
        y_true=y,
        y_pred=y,
        sensitive_features=groupes
    )

    # Affichage des résultats par groupe
    print("\n" + "=" * 60)
    print("PARITÉ SELON :", col)
    display(mf.by_group.round(3))

    # Calcul de l'écart et du ratio de parité démographique
    ecart = demographic_parity_difference(
        y_true=y,
        y_pred=y,
        sensitive_features=groupes
    )

    ratio = demographic_parity_ratio(
        y_true=y,
        y_pred=y,
        sensitive_features=groupes
    )

    # Enregistrement des résultats
    synthese.append({
        "variable": col,
        "écart": ecart,
        "ratio": ratio
    })

# Tableau récapitulatif
synthese = pd.DataFrame(synthese)

print("\nSYNTHÈSE DE LA PARITÉ")
display(synthese.round(3))



PARITÉ SELON : tranche_age


,Effectif,Taux d'accord
tranche_age,,
18-24,1788.0,0.183
25-34,5299.0,0.440
35-44,5228.0,0.689
45-54,4002.0,0.829
55-64,2636.0,0.873
65+,1431.0,0.813



PARITÉ SELON : genre


,Effectif,Taux d'accord
genre,,
F,8731.0,0.596
H,11653.0,0.672



PARITÉ SELON : region


,Effectif,Taux d'accord
region,,
Nord-Est,4735.0,0.617
Ouest,4321.0,0.626
Outre-mer,207.0,0.575
Sud-Est,4499.0,0.625
Sud-Ouest,2825.0,0.619
Île-de-France,3797.0,0.721



SYNTHÈSE DE LA PARITÉ


,variable,écart,ratio
0,tranche_age,0.690,0.210
1,genre,0.076,0.887
2,region,0.146,0.798


### Interprétation – Parité de la décision

**Tranche d'âge :** Les taux d'accord présentent de fortes différences selon l'âge. Les 18-24 ans ont le taux le plus faible (18,3 %), contre 87,3 % pour les 55-64 ans. Le ratio de 0,21 révèle un écart important entre les groupes.

**Genre :** Les hommes obtiennent davantage de crédits (67,2 %) que les femmes (59,6 %). L'écart reste relativement limité, avec un ratio de 0,887.

**Région :** L'Île-de-France présente le taux d'accord le plus élevé (72,1 %), contre 57,5 % en Outre-mer. Le ratio de 0,798 montre une différence à surveiller.

**Conclusion :** Les disparités les plus importantes concernent l'âge, suivi de la région et du genre. Ces écarts méritent une analyse complémentaire pour comprendre les différences dans les décisions d'octroi de crédit.

## 3. Les croisements
Un biais peut se cacher dans un croisement. Calculez le taux par combinaison de deux variables sensibles.

In [8]:

# 3. Croisement de deux variables sensibles

# Création des groupes combinés : âge et genre
groupes_croises = pd.DataFrame({
    "tranche_age": df["tranche_age"].fillna("inconnu"),
    "genre": df["genre"].fillna("inconnu")
})

# Calcul du taux d'accord et de l'effectif pour chaque combinaison
mf2 = MetricFrame(
    metrics={
        "Effectif": count,
        "Taux d'accord": selection_rate
    },
    y_true=y,
    y_pred=y,
    sensitive_features=groupes_croises
)

# Affichage des résultats
display(mf2.by_group.round(3))


Effectif  Taux d'accord
tranche_age genre                         
18-24       F         713.0          0.164
            H        1075.0          0.195
25-34       F        2354.0          0.386
            H        2945.0          0.484
35-44       F        2190.0          0.636
            H        3038.0          0.726
45-54       F        1670.0          0.787
            H        2332.0          0.858
55-64       F        1158.0          0.844
            H        1478.0          0.896
65+         F         646.0          0.771
            H         785.0          0.848

### Interprétation – Croisement âge et genre

Le croisement entre l'âge et le genre montre que les hommes ont systématiquement un taux d'accord supérieur à celui des femmes, quelle que soit la tranche d'âge. L'écart le plus important concerne les 25-34 ans, avec 9,8 points de différence. Les jeunes de 18-24 ans présentent les taux d'accord les plus faibles pour les deux genres.

**Conclusion :** les différences observées selon le genre sont présentes dans toutes les tranches d'âge. Le croisement met donc en évidence des disparités qui méritent d'être approfondies.

## 4. Les corrélations induites
La disparité la plus forte persiste-t-elle **à segment égal** ? Comparez les taux par variable sensible à l'intérieur de chaque segment métier.

In [11]:
# ✍️ À VOUS : remplacez les ... et traitez les TODO
var = synthese.sort_values("ratio")["variable"].iloc[0]


# 4. Les corrélations induites

# Sélection de la variable présentant le plus faible ratio
var = synthese.sort_values("ratio")["variable"].iloc[0]

print("Variable sensible étudiée :", var)

# Identification du segment métier
segment = cfg["segment"]

# Création des groupes de segments
segments = df[segment].fillna("inconnu")

# Comparaison des taux d'accord à l'intérieur de chaque segment
for modalite in segments.unique():

    masque = segments == modalite

    y_segment = df.loc[masque, cfg["decision"]]
    groupe = df.loc[masque, var].fillna("inconnu")

    mf_segment = MetricFrame(
        metrics={
            "Effectif": count,
            "Taux d'accord": selection_rate
        },
        y_true=y_segment,
        y_pred=y_segment,
        sensitive_features=groupe
    )

    print("\nSegment :", modalite)
    print("Variable comparée :", var)
    display(mf_segment.by_group.round(3))


Variable sensible étudiée : tranche_age

Segment : personnel
Variable comparée : tranche_age


,Effectif,Taux d'accord
tranche_age,,
18-24,537.0,0.166
25-34,1605.0,0.459
35-44,1571.0,0.707
45-54,1184.0,0.828
55-64,806.0,0.876
65+,414.0,0.836



Segment : travaux
Variable comparée : tranche_age


,Effectif,Taux d'accord
tranche_age,,
18-24,460.0,0.224
25-34,1360.0,0.468
35-44,1294.0,0.704
45-54,980.0,0.856
55-64,684.0,0.882
65+,361.0,0.814



Segment : auto
Variable comparée : tranche_age


,Effectif,Taux d'accord
tranche_age,,
18-24,536.0,0.175
25-34,1567.0,0.403
35-44,1593.0,0.655
45-54,1196.0,0.802
55-64,768.0,0.858
65+,422.0,0.810



Segment : renouvelable
Variable comparée : tranche_age


,Effectif,Taux d'accord
tranche_age,,
18-24,255.0,0.161
25-34,767.0,0.426
35-44,770.0,0.695
45-54,642.0,0.838
55-64,378.0,0.881
65+,234.0,0.778


### Interprétation – Comparaison à segment égal

Les 18-24 ans conservent les taux d'accord les plus faibles, tandis que les 55-64 ans présentent les taux les plus élevés.

Ces résultats montrent que le type de crédit ne suffit pas à expliquer les différences observées selon l'âge.

**Conclusion :** les disparités liées à l'âge restent importantes même en tenant compte du type de crédit demandé. D'autres caractéristiques pourraient également expliquer ces différences.

**Lecture** : la disparité vient-elle de la variable sensible elle-même, ou d'un proxy ?
Les écarts de taux d'accord selon l'âge persistent même lorsque l'on compare les clients ayant le même type de crédit. Le type de crédit ne suffit donc pas à expliquer ces disparités. Cependant, d'autres variables, comme le revenu ou le taux d'endettement, pourraient également influencer les décisions.

Conclusion : les différences liées à l'âge restent visibles, mais il est nécessaire d'étudier d'autres variables pour déterminer si elles sont directement liées à l'âge ou à des facteurs indirects (proxies)

## 5. L'évolution dans le temps
La disparité a-t-elle changé avec l'événement du brief ? Comparez les taux avant et après la date.

In [13]:
# ✍️ À VOUS : remplacez les ... et traitez les TODO


# 5. Évolution des disparités dans le temps

# Date de déploiement du score V2
date_evenement = pd.Timestamp("2024-03-01")

# Conversion de la date de demande
df["date_demande"] = pd.to_datetime(df["date_demande"])

# Séparation des périodes
avant = df["date_demande"] < date_evenement
apres = df["date_demande"] >= date_evenement

# Comparaison des taux d'accord par tranche d'âge
for nom, masque in [
    ("Avant le déploiement", avant),
    ("Après le déploiement", apres)
]:

    print("\n" + "=" * 55)
    print(nom)

    y_periode = df.loc[masque, cfg["decision"]]
    groupes = df.loc[masque, var].fillna("inconnu")

    mf_temps = MetricFrame(
        metrics={
            "Effectif": count,
            "Taux d'accord": selection_rate
        },
        y_true=y_periode,
        y_pred=y_periode,
        sensitive_features=groupes
    )

    display(mf_temps.by_group.round(3))




Avant le déploiement


,Effectif,Taux d'accord
tranche_age,,
18-24,668.0,0.361
25-34,1975.0,0.603
35-44,1927.0,0.814
45-54,1494.0,0.912
55-64,998.0,0.940
65+,523.0,0.897



Après le déploiement


,Effectif,Taux d'accord
tranche_age,,
18-24,1120.0,0.077
25-34,3324.0,0.343
35-44,3301.0,0.615
45-54,2508.0,0.779
55-64,1638.0,0.832
65+,908.0,0.765


### Interprétation – Évolution dans le temps

Oui, les disparités ont évolué après le déploiement du score V2 en mars 2024. Les taux d'accord ont diminué pour toutes les tranches d'âge. La baisse est particulièrement forte chez les 18-24 ans, passant de 36,1 % avant le déploiement à 7,7 % après, soit une diminution de 28,4 points.

Les écarts entre les tranches d'âge se sont donc accentués, les jeunes restant les moins favorisés.

**Conclusion :** les disparités liées à l'âge se sont renforcées après le déploiement du score V2. Cependant, ces résultats ne permettent pas d'affirmer que le nouveau score est directement responsable de cette évolution.

## 6. L'égalité des chances
Quand le résultat réel est connu, comparez le **taux de vrais positifs** par groupe : parmi ceux pour qui le résultat a été favorable, quelle part avait reçu la décision favorable ?

In [14]:
# ✍️ À VOUS : remplacez les ... et traitez les TODO
connu = df[cfg["resultat"]].notna()


# 6. L'égalité des chances

# On conserve uniquement les résultats réels connus
connu = df[cfg["resultat"]].notna()

# Résultat réel et décision de crédit
y_reel = df.loc[connu, cfg["resultat"]]
y_decision = df.loc[connu, cfg["decision"]]

# Variable sensible étudiée : tranche d'âge
groupes = df.loc[connu, var].fillna("inconnu")

# Calcul du taux de vrais positifs par groupe
mf_chances = MetricFrame(
    metrics={
        "Effectif": count,
        "Taux de vrais positifs": true_positive_rate
    },
    y_true=y_reel,
    y_pred=y_decision,
    sensitive_features=groupes
)

print("Égalité des chances selon :", var)
display(mf_chances.by_group.round(3))


Égalité des chances selon : tranche_age


,Effectif,Taux de vrais positifs
tranche_age,,
18-24,327.0,1.0
25-34,2332.0,1.0
35-44,3600.0,1.0
45-54,3316.0,1.0
55-64,2301.0,1.0
65+,1164.0,1.0


### Interprétation – Égalité des chances

Le taux de vrais positifs est de 100 % pour toutes les tranches d’âge. Cependant, ce résultat vient du fait que les remboursements sont connus uniquement pour les personnes ayant obtenu un crédit.

Dans les données utilisées pour ce calcul, toutes les décisions sont donc favorables. Le résultat de 100 % est mécanique : il ne permet pas de savoir si des personnes refusées auraient pu rembourser sans incident.

**Conclusion :** cet indicateur ne permet pas d’évaluer l’égalité des chances entre les tranches d’âge. Nous pouvons mesurer les différences de taux d’accord, mais pas déterminer quels demandeurs refusés auraient remboursé leur crédit.

## 7. Les limites de votre analyse
Listez au moins trois limites : données, référence, méthode.

### 7. Les limites de notre analyse

Notre analyse présente plusieurs limites :

**1. Limites des données :** certaines informations sont manquantes, notamment les résultats de remboursement pour les crédits refusés. Cela limite l'évaluation réelle de la capacité de remboursement de tous les clients.

**2. Limites de la référence :** la population de référence ne correspond pas forcément exactement à la population des clients susceptibles de demander un crédit chez Crédit Azur. Les différences de représentativité doivent donc être interprétées avec prudence.

**3. Limites de la méthode :** les écarts observés ne prouvent pas automatiquement une discrimination. D'autres variables, comme le revenu, l'endettement ou la situation professionnelle, peuvent influencer les décisions.

**4. Limites de l'analyse temporelle :** la baisse des taux d'accord après mars 2024 ne permet pas d'affirmer que le déploiement du score V2 en est directement responsable.

**5. Limites de l’égalité des chances :** le taux de vrais positifs de 100 % est mécanique, car les remboursements sont connus uniquement pour les crédits accordés. Il ne permet donc pas de conclure à une égalité des chances entre les groupes.

## 💾 Livrable de fin de journée
Votre notebook d'analyse complet, chaque choix justifié dans une cellule markdown, enregistré dans GitHub :
*Fichier › Enregistrer une copie dans GitHub*, message « Jour 1 : analyse et biais ». Envoyez le lien du dépôt au formateur.